# **1. Descrição e coleta de dados**

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
spotify_songs = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv')

In [ ]:
spotify_songs.head(3)

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616


# **3. Selecionando os dados adequados**

**3.1 Removendo linhas duplicadas**

In [ ]:
# Encontra todas as linhas que compartilham o mesmo track_id
duplicates_only = spotify_songs[spotify_songs.duplicated(subset=["track_id"], keep=False)]

# Ordena eles pelo track_id
duplicates_sorted = duplicates_only.sort_values("track_id")

# Imprime as primeiras 10 linhas
print(duplicates_sorted[["track_name", "playlist_genre", "danceability", "energy"]].head(10))

                            track_name playlist_genre  danceability  energy
32084    Hide Away (feat. Envy Monroe)            edm         0.573   0.746
28696    Hide Away (feat. Envy Monroe)            edm         0.573   0.746
23850       We Own It (Fast & Furious)            r&b         0.554   0.899
28968       We Own It (Fast & Furious)            edm         0.554   0.899
9387        We Own It (Fast & Furious)            rap         0.554   0.899
7853   Ain't No Future In Yo' Frontin'            rap         0.672   0.761
9345   Ain't No Future In Yo' Frontin'            rap         0.672   0.761
3124                               Hot            pop         0.607   0.908
14626                              Hot           rock         0.607   0.908
18487                     La Mordidita          latin         0.725   0.903


In [ ]:
linhas_antes = len(spotify_songs)
print(f"Linhas antes: {linhas_antes}")

# Remove os duplicados baseados no ID, mantendo o primeiro encontrado
spotify_songs = spotify_songs.drop_duplicates(subset=["track_id"], keep="first")

print(f"Linhas depois: {len(spotify_songs)}")
print(f"Total de duplicados removidos: {linhas_antes - len(spotify_songs)}")

Linhas antes: 32833
Linhas depois: 28356
Total de duplicados removidos: 4477


**3.2 Arrumando as datas**

In [ ]:
# Busca por erros na formatacao da data

# Tenta converter para datetime, forcando os erros a se tornarem 'NaT'
test_dates = pd.to_datetime(spotify_songs["track_album_release_date"], format="%Y-%m-%d", errors="coerce")

# Encontra as linhas onde a conversao falhou
formatting_errors = spotify_songs[test_dates.isna()]

print(formatting_errors["track_album_release_date"])

151      2012
749      1998
750      1996
751      1999
753      1993
         ... 
32360    2013
32767    2012
32774    2011
32775    2011
32827    2013
Name: track_album_release_date, Length: 1681, dtype: object


In [ ]:
# Formata a data para que contenha apenas o ano, criando a coluna "release_year"
# Remove a antiga coluna "track_album_release_date"

# 'format="mixed"' tells pandas to figure out if it's "YYYY-MM-DD" or just "YYYY"
spotify_songs["release_year"] = pd.to_datetime(spotify_songs["track_album_release_date"], format="mixed").dt.year

spotify_songs = spotify_songs.drop(columns=["track_album_release_date"])

# Imprime os anos em que as musicas do dataset foram lancados
spotify_songs["release_year"].unique()

array([2019, 2018, 2017, 2016, 2014, 2012, 2015, 2013, 2011, 2010, 2008,
       2020, 2007, 1998, 1996, 1999, 2009, 1993, 1995, 1991, 2000, 1994,
       1992, 1997, 2001, 2006, 2002, 2003, 2005, 1990, 2004, 1988, 1984,
       1982, 1973, 1979, 1977, 1981, 1974, 1970, 1976, 1987, 1978, 1969,
       1986, 1980, 1983, 1985, 1989, 1975, 1968, 1971, 1972, 1967, 1966,
       1965, 1964, 1963, 1962, 1957, 1958, 1961, 1960], dtype=int32)

**3.3 Removendo os dados nulos**

In [ ]:
# Busca por dados nulos no dataset inteiro
spotify_songs.isnull().sum()

# Cria um filtro que procura por valores nulos
null_filter = spotify_songs.isnull().any(axis=1)

# Aplica o filtro para ver as linhas
rows_with_nulls = spotify_songs[null_filter]

print(rows_with_nulls)

                     track_id track_name track_artist  track_popularity  \
8151   69gRFGOWY9OMpFJgFol1u0        NaN          NaN                 0   
9282   5cjecvX0CmC9gK0Laf5EMQ        NaN          NaN                 0   
9283   5TTzhRSWQS4Yu8xTgAuq6D        NaN          NaN                 0   
19568  3VKFip3OdAvv4OfNTgFWeQ        NaN          NaN                 0   

               track_album_id track_album_name       playlist_name  \
8151   717UG2du6utFe7CdmpuUe3              NaN             HIP&HOP   
9282   3luHJEPw434tvNbme3SP8M              NaN         GANGSTA Rap   
9283   3luHJEPw434tvNbme3SP8M              NaN         GANGSTA Rap   
19568  717UG2du6utFe7CdmpuUe3              NaN  Reggaeton viejito🔥   

                  playlist_id playlist_genre playlist_subgenre  ...  loudness  \
8151   5DyJsJZOpMJh34WvUrQzMV            rap  southern hip hop  ...    -7.635   
9282   5GA8GDo7RQC3JEanT81B3g            rap      gangster rap  ...    -5.364   
9283   5GA8GDo7RQC3JEanT81B3g 

In [ ]:
# Remove qualquer linha que contenha pelo menos um NaN
spotify_songs = spotify_songs.dropna()

# Reseta o indice para que o numero das linhas seja perfeitamente sequencial
spotify_songs = spotify_songs.reset_index(drop=True)

spotify_songs.shape

(28352, 23)

**3.4 Removendo colunas inuteis**

In [ ]:
# Remove os textos e os ids
columns_to_drop = [
    "track_id", "track_album_id", "playlist_id",
    "track_name", "track_album_name", "playlist_name", "track_artist"
]

dados_ml = spotify_songs.drop(columns=columns_to_drop)

In [ ]:
dados_ml.head(3)

,track_popularity,playlist_genre,playlist_subgenre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,release_year
0,66,pop,dance pop,0.748,0.916,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754,2019
1,67,pop,dance pop,0.726,0.815,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600,2019
2,70,pop,dance pop,0.675,0.931,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616,2019


# **2. Analise dos dados**

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
import plotly.express as px

# Agrupa os dados por ano e calcula a media do loudness
avg_loudness = spotify_songs.groupby("release_year")["loudness"].mean().reset_index()

# Cria o grafico
fig = px.line(
    avg_loudness,
    x="release_year",
    y="loudness",
    markers=True,
    title="Variação Média do Loudness ao Longo dos Anos"
)
fig.show()

In [ ]:
# 1. Group by year and get the mean for all numeric columns
# Add numeric_only=True to prevent the error
yearly_averages = spotify_songs.groupby("release_year").mean(numeric_only=True).reset_index()

fig = go.Figure()

# Notice we changed y = ["acousticness"] to y = spotify_songs["acousticness"]
fig.add_trace(go.Scatter(x = yearly_averages["release_year"], y = yearly_averages["acousticness"], name = "Acousticness"))
fig.add_trace(go.Scatter(x = yearly_averages["release_year"], y = yearly_averages["valence"], name = "Valence"))
fig.add_trace(go.Scatter(x = yearly_averages["release_year"], y = yearly_averages["danceability"], name = "Danceability"))
fig.add_trace(go.Scatter(x = yearly_averages["release_year"], y = yearly_averages["energy"], name = "Energy"))
fig.add_trace(go.Scatter(x = yearly_averages["release_year"], y = yearly_averages["instrumentalness"], name = "Instrumentalness"))
fig.add_trace(go.Scatter(x = yearly_averages["release_year"], y = yearly_averages["liveness"], name = "Liveness"))
fig.add_trace(go.Scatter(x = yearly_averages["release_year"], y = yearly_averages["speechiness"], name = "Speechiness"))

fig.show()

### **2.1 Visualizando as Correlações Mais Fortes**
Vamos olhar mais de perto para as variáveis que apresentaram maior correlação no heatmap.

In [ ]:
import plotly.express as px

# Creating a scatter plot for Valence vs Danceability
fig3 = px.scatter(
    spotify_songs.sample(2000),
    x="valence",
    y="danceability",
    trendline="ols",
    title="Correlação: Valence vs Danceability (Amostra de 2000 músicas)",
    opacity=0.5,
    color_discrete_sequence=['#9467bd']
)
fig3.show()

In [ ]:
correlation_val_dance = spotify_songs['valence'].corr(spotify_songs['danceability'])
print(f'Correlation coefficient between Valence and Danceability: {correlation_val_dance:.4f}')

Correlation coefficient between Valence and Danceability: 0.3338


In [ ]:
correlation_energy_loudness = spotify_songs['energy'].corr(spotify_songs['loudness'])
print(f'Correlation coefficient between Energy and Loudness: {correlation_energy_loudness:.4f}')

Correlation coefficient between Energy and Loudness: 0.6822


In [ ]:
import plotly.express as px

# Define quais colunas vao aparecer no heatmap
features = ["danceability", "energy", "loudness", "speechiness", "mode", "release_year",
            "acousticness", "instrumentalness", "liveness", "valence", "tempo", "duration_ms", "key", "track_popularity"]

# Pega apenas as colunas selecionadas e depois calcula a correlacao
correlation_matrix = spotify_songs[features].corr()

# Cria a matriz
fig = px.imshow(
    correlation_matrix,
    text_auto=".2f", # Arrendonda para duas casas decimais
    aspect="auto",
    title="Correlation Heatmap of Spotify Audio Features"
)
fig.show()

In [ ]:
import plotly.express as px

# Criando um gráfico de dispersão (scatter) para a maior correlação: Energy vs Loudness
fig1 = px.scatter(
    spotify_songs.sample(2000), # Usando uma amostra para manter o gráfico performático
    x="energy",
    y="loudness",
    trendline="ols",
    title="Correlação Positiva: Energy vs Loudness (Amostra de 2000 músicas)",
    opacity=0.5,
    color_discrete_sequence=['#1DB954']
)
fig1.show()

# Criando um gráfico para a correlação negativa: Energy vs Acousticness
fig2 = px.scatter(
    spotify_songs.sample(2000),
    x="energy",
    y="acousticness",
    trendline="ols",
    title="Correlação Negativa: Energy vs Acousticness (Amostra de 2000 músicas)",
    opacity=0.5,
    color_discrete_sequence=['#ff4b4b']
)
fig2.show()

In [ ]:
import plotly.express as px

# Conta quantas vezes cada ano aparece no dataset
contagem_anos = spotify_songs["release_year"].value_counts().reset_index()

# Renomeia as colunas para ficar mais fácil de entender
contagem_anos.columns = ["Ano", "Quantidade de Músicas"]

# Ordena os anos por ordem cronologica
contagem_anos = contagem_anos.sort_values("Ano")

# Cria o grafico de barras
fig = px.bar(
    contagem_anos,
    x="Ano",
    y="Quantidade de Músicas",
    title="Distribuição da Quantidade de Músicas por Ano",
    text_auto=True # Mostra o número exato de músicas no topo de cada barra!
)

# Da um zoom no visual para ficar mais bonito
fig.update_layout(xaxis_title="Ano de Lançamento", yaxis_title="Total de Músicas", width=900, height=500)

fig.show()

In [ ]:
  import plotly.express as px

# 1. Lista de todas as variáveis numéricas de áudio
features_to_plot = ["danceability", "energy", "loudness", "speechiness",
                    "acousticness", "instrumentalness", "liveness", "valence", "tempo"]

# 2. Loop para criar um histograma para cada característica
for feature in features_to_plot:
    fig = px.histogram(
        spotify_songs,
        x=feature,
        nbins=50, # Define a quantidade de barras para deixar o gráfico detalhado
        marginal="box", # PRO-TIP: Adiciona um mini-boxplot no topo para ver os outliers!
        title=f"Distribuição Geral de {feature.capitalize()} por Gênero",
        color='playlist_genre' # Color bars by playlist_genre
    )

    fig.update_layout(width=800, height=400)
    fig.show()

In [ ]:
import plotly.express as px

# Count the occurrences of each genre
genre_counts = spotify_songs['playlist_genre'].value_counts().reset_index()
genre_counts.columns = ['Genre', 'Count']

# Create a bar chart for genre distribution
fig = px.bar(
    genre_counts,
    x='Genre',
    y='Count',
    title='Distribution of Songs by Playlist Genre',
    color='Genre', # Color bars by genre
    text_auto=True # Display count on top of bars
)

fig.update_layout(
    xaxis_title='Playlist Genre',
    yaxis_title='Number of Songs',
    width=900,
    height=500
)

fig.show()